# Model 1: PPG → Blood Pressure Estimation
**Dataset**: VitalDB (PhysioNet)

This notebook trains a deep learning model to estimate Systolic (SBP) and Diastolic (DBP) blood pressure from PPG waveform segments.

**Architecture**: 1D CNN + BiLSTM + Attention

**Input**: 625-sample PPG window (5 seconds @ 125 Hz)

**Output**: [SBP, DBP] in mmHg

In [ ]:
# Install dependencies
! pip install vitaldb wfdb torch torchvision torchaudio scikit-learn matplotlib numpy pandas tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import vitaldb
from tqdm import tqdm
import warnings
import os
import pickle

warnings.filterwarnings('ignore')

# ─── Device ───────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Data Loading from VitalDB

In [ ]:
# ─── Configuration ─────────────────────────────────────────────────────────────
N_CASES      = 200      # Number of VitalDB cases to load (increase for better results)
FS           = 125      # Target sampling frequency (Hz)
WIN_SEC      = 5        # Window size in seconds
WIN_SAMPLES  = FS * WIN_SEC  # 625 samples
STRIDE_SEC   = 2        # Stride between windows
STRIDE       = FS * STRIDE_SEC

PPG_TRACK    = 'SNUADC/PLETH'     # PPG waveform track
SBP_TRACK    = 'Solar8000/ART_SBP'  # Systolic BP numeric
DBP_TRACK    = 'Solar8000/ART_DBP'  # Diastolic BP numeric

# Valid BP range (filter artifacts)
SBP_MIN, SBP_MAX = 70, 200
DBP_MIN, DBP_MAX = 40, 130

In [ ]:
def load_vitaldb_data(n_cases=N_CASES):
    """
    Load PPG and BP data from VitalDB cloud API.
    Returns lists of (ppg_windows, sbp_values, dbp_values).
    """
    all_ppg = []
    all_sbp = []
    all_dbp = []

    case_ids = list(range(1, n_cases + 1))
    print(f'Loading {n_cases} cases from VitalDB...')

    for cid in tqdm(case_ids):
        try:
            # Load PPG at 125 Hz, BP numerics at 1 Hz
            ppg_data  = vitaldb.load_case(cid, PPG_TRACK, interval=1/FS)
            sbp_data  = vitaldb.load_case(cid, SBP_TRACK, interval=1)
            dbp_data  = vitaldb.load_case(cid, DBP_TRACK, interval=1)

            if ppg_data is None or sbp_data is None or dbp_data is None:
                continue

            ppg_arr = ppg_data.flatten()
            sbp_arr = sbp_data.flatten()
            dbp_arr = dbp_data.flatten()

            # Slide window over PPG
            n_windows = (len(ppg_arr) - WIN_SAMPLES) // STRIDE
            for w in range(n_windows):
                start = w * STRIDE
                end   = start + WIN_SAMPLES
                ppg_win = ppg_arr[start:end]

                # Corresponding BP (center of window in seconds)
                t_center = (start + WIN_SAMPLES // 2) // FS
                if t_center >= len(sbp_arr) or t_center >= len(dbp_arr):
                    continue

                sbp_val = sbp_arr[t_center]
                dbp_val = dbp_arr[t_center]

                # Quality checks
                if np.any(np.isnan(ppg_win)) or np.isnan(sbp_val) or np.isnan(dbp_val):
                    continue
                if not (SBP_MIN <= sbp_val <= SBP_MAX and DBP_MIN <= dbp_val <= DBP_MAX):
                    continue
                if np.std(ppg_win) < 1e-4:  # flat signal
                    continue

                all_ppg.append(ppg_win.astype(np.float32))
                all_sbp.append(float(sbp_val))
                all_dbp.append(float(dbp_val))

        except Exception:
            continue

    print(f'Total windows collected: {len(all_ppg)}')
    return np.array(all_ppg), np.array(all_sbp), np.array(all_dbp)


# ─── Load (or from cache) ──────────────────────────────────────────────────────
CACHE_FILE = 'ppg_bp_cache.pkl'
if os.path.exists(CACHE_FILE):
    print('Loading from cache...')
    with open(CACHE_FILE, 'rb') as f:
        ppg_windows, sbp_labels, dbp_labels = pickle.load(f)
else:
    ppg_windows, sbp_labels, dbp_labels = load_vitaldb_data(N_CASES)
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump((ppg_windows, sbp_labels, dbp_labels), f)

print(f'PPG shape: {ppg_windows.shape}')
print(f'SBP range: {sbp_labels.min():.1f} – {sbp_labels.max():.1f} mmHg')
print(f'DBP range: {dbp_labels.min():.1f} – {dbp_labels.max():.1f} mmHg')

## 2. Preprocessing & Dataset

In [ ]:
# ─── Normalize PPG per window (zero mean, unit std) ────────────────────────────
def normalize_ppg(x):
    mu  = x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True) + 1e-8
    return (x - mu) / std

ppg_norm = normalize_ppg(ppg_windows)   # (N, 625)

# ─── Normalize BP labels ───────────────────────────────────────────────────────
bp_labels   = np.stack([sbp_labels, dbp_labels], axis=1)  # (N, 2)
bp_scaler   = StandardScaler()
bp_norm     = bp_scaler.fit_transform(bp_labels).astype(np.float32)

# Save scaler for later use in Model 3
import joblib
joblib.dump(bp_scaler, 'bp_scaler.pkl')

print(f'Normalized PPG  shape: {ppg_norm.shape}')
print(f'Normalized BP   shape: {bp_norm.shape}')

In [ ]:
# ─── Plot sample PPG and BP ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
t = np.arange(WIN_SAMPLES) / FS
axes[0].plot(t, ppg_norm[0])
axes[0].set_title('Sample Normalized PPG Window')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude (normalized)')

axes[1].hist(sbp_labels, bins=50, alpha=0.7, label='SBP', color='red')
axes[1].hist(dbp_labels, bins=50, alpha=0.7, label='DBP', color='blue')
axes[1].set_title('BP Distribution')
axes[1].set_xlabel('mmHg')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
class PPGDataset(Dataset):
    def __init__(self, ppg, bp):
        self.ppg = torch.tensor(ppg[:, np.newaxis, :], dtype=torch.float32)  # (N,1,625)
        self.bp  = torch.tensor(bp, dtype=torch.float32)                      # (N,2)

    def __len__(self):
        return len(self.ppg)

    def __getitem__(self, idx):
        return self.ppg[idx], self.bp[idx]


dataset    = PPGDataset(ppg_norm, bp_norm)
n_total    = len(dataset)
n_train    = int(0.70 * n_total)
n_val      = int(0.15 * n_total)
n_test     = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=256, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')

## 3. Model Architecture: CNN-BiLSTM with Attention

In [ ]:
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style temporal attention."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):  # x: (B, C, T)
        w = x.mean(dim=2)       # global average pooling → (B, C)
        w = self.fc(w)          # → (B, C)
        return x * w.unsqueeze(2)


class ResBlock1D(nn.Module):
    """1D Residual block with attention."""
    def __init__(self, channels, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size, padding=pad),
            nn.BatchNorm1d(channels),
        )
        self.attn = ChannelAttention(channels)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.attn(self.conv(x)) + x)


class PPGtoBP(nn.Module):
    """
    1D CNN feature extractor → BiLSTM temporal modelling → MLP regressor.
    Input : (B, 1, 625)
    Output: (B, 2)  [SBP_norm, DBP_norm]
    """
    def __init__(self, win_len=625, lstm_hidden=128, lstm_layers=2, dropout=0.3):
        super().__init__()

        # ── Stem ──────────────────────────────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, padding=7, stride=2),  # → 312
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        # ── Residual CNN blocks ───────────────────────────────────────────────
        self.layer1 = nn.Sequential(
            nn.Conv1d(32, 64, 3, stride=2, padding=1),  # → 156
            ResBlock1D(64),
            ResBlock1D(64),
        )
        self.layer2 = nn.Sequential(
            nn.Conv1d(64, 128, 3, stride=2, padding=1),  # → 78
            ResBlock1D(128),
            ResBlock1D(128),
        )
        self.layer3 = nn.Sequential(
            nn.Conv1d(128, 256, 3, stride=2, padding=1),  # → 39
            ResBlock1D(256),
        )

        # ── BiLSTM ────────────────────────────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=256, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            bidirectional=True, dropout=dropout
        )

        # ── Self-attention over LSTM outputs ──────────────────────────────────
        self.query = nn.Linear(lstm_hidden * 2, 1)

        # ── Regressor ─────────────────────────────────────────────────────────
        self.regressor = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):           # x: (B,1,625)
        x = self.stem(x)            # (B,32,312)
        x = self.layer1(x)          # (B,64,156)
        x = self.layer2(x)          # (B,128,78)
        x = self.layer3(x)          # (B,256,39)
        x = x.permute(0, 2, 1)      # (B,39,256) – time-first for LSTM
        x, _ = self.bilstm(x)       # (B,39,256)
        # Attention-weighted pooling
        w = torch.softmax(self.query(x), dim=1)  # (B,39,1)
        x = (x * w).sum(dim=1)      # (B,256)
        return self.regressor(x)    # (B,2)


model = PPGtoBP().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 4. Training

In [ ]:
EPOCHS    = 50
LR        = 1e-3
PATIENCE  = 10

criterion  = nn.SmoothL1Loss()   # Huber loss – robust to outliers
optimizer  = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0
    with torch.set_grad_enabled(train):
        for ppg, bp in loader:
            ppg, bp = ppg.to(device), bp.to(device)
            pred = model(ppg)
            loss = criterion(pred, bp)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item() * len(ppg)
    return total_loss / len(loader.dataset)


train_losses, val_losses = [], []
best_val = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss  = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader,   train=False)
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), 'best_ppg_to_bp.pth')
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 5 == 0 or patience_counter == 0:
        print(f'Epoch {epoch:3d} | Train: {tr_loss:.4f} | Val: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'Best val loss: {best_val:.4f}')

## 5. Evaluation

In [ ]:
# ─── Loss curves ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Huber Loss')
plt.title('Training Curves – PPG → BP')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ─── Load best weights and evaluate on test set ───────────────────────────────
model.load_state_dict(torch.load('best_ppg_to_bp.pth', map_location=device))
model.eval()

preds_norm, trues_norm = [], []
with torch.no_grad():
    for ppg, bp in test_loader:
        ppg = ppg.to(device)
        pred = model(ppg).cpu().numpy()
        preds_norm.append(pred)
        trues_norm.append(bp.numpy())

preds_norm = np.vstack(preds_norm)
trues_norm = np.vstack(trues_norm)

# Inverse-transform to mmHg
preds_mmhg = bp_scaler.inverse_transform(preds_norm)
trues_mmhg = bp_scaler.inverse_transform(trues_norm)

for i, name in enumerate(['SBP', 'DBP']):
    mae  = mean_absolute_error(trues_mmhg[:, i], preds_mmhg[:, i])
    rmse = np.sqrt(mean_squared_error(trues_mmhg[:, i], preds_mmhg[:, i]))
    corr = np.corrcoef(trues_mmhg[:, i], preds_mmhg[:, i])[0, 1]
    print(f'{name} → MAE: {mae:.2f} mmHg | RMSE: {rmse:.2f} mmHg | Pearson r: {corr:.4f}')

In [ ]:
# ─── Scatter plots ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['red', 'blue']
names  = ['SBP', 'DBP']

for i, (ax, name, c) in enumerate(zip(axes, names, colors)):
    ax.scatter(trues_mmhg[:, i], preds_mmhg[:, i], alpha=0.3, s=5, color=c)
    lim = [min(trues_mmhg[:, i].min(), preds_mmhg[:, i].min()),
           max(trues_mmhg[:, i].max(), preds_mmhg[:, i].max())]
    ax.plot(lim, lim, 'k--', label='Ideal')
    ax.set_xlabel(f'True {name} (mmHg)')
    ax.set_ylabel(f'Predicted {name} (mmHg)')
    ax.set_title(f'{name} Prediction')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nModel saved as: best_ppg_to_bp.pth')
print('BP scaler saved as: bp_scaler.pkl')

## 6. Inference Helper (used by Model 3)

In [ ]:
def predict_bp_from_ppg(ppg_window: np.ndarray, model=model, scaler=bp_scaler) -> dict:
    """
    Given a raw PPG window (625 samples), return SBP and DBP in mmHg.
    """
    # Normalize
    mu  = ppg_window.mean()
    std = ppg_window.std() + 1e-8
    x   = (ppg_window - mu) / std

    x_t = torch.tensor(x[np.newaxis, np.newaxis, :], dtype=torch.float32).to(device)
    with torch.no_grad():
        pred_norm = model(x_t).cpu().numpy()
    sbp, dbp = scaler.inverse_transform(pred_norm)[0]
    return {'SBP': round(float(sbp), 1), 'DBP': round(float(dbp), 1)}


# Quick smoke-test
sample_ppg = ppg_windows[0]
result = predict_bp_from_ppg(sample_ppg)
true_sbp = sbp_labels[0]
true_dbp = dbp_labels[0]
print(f'Sample prediction: SBP={result["SBP"]} mmHg, DBP={result["DBP"]} mmHg')
print(f'True values:       SBP={true_sbp:.1f} mmHg, DBP={true_dbp:.1f} mmHg')